# Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt

# Get CIFAR-10 Data

In [ ]:
# Define the transformations to be applied to the images (if any).
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomCrop(224, padding=4),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Testing: No noise, just normalize
test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Get the training set and load it using DataLoader.
trainset=torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)

# Get the test set and load it using DataLoader.
testset=torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=4)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(len(classes))

## View the images

In [ ]:
# Function to show an image
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.figure(figsize=(10, 10))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.show()

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:4], nrow=4))

# Print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

# Make ResNet-18

In [ ]:
# Import Libraries
import torch
import torch.nn as nn

# Make the Neural Network

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self,in_channels, out_channels,stride=1, downsample=None):
        super(ResidualBlock,self).__init__()
        # First 3x3 convolutional layer
        self.conv1=nn.Conv2d(in_channels, out_channels, kernel_size=3,stride=stride, padding=1, bias=False)
        self.bn1=nn.BatchNorm2d(out_channels)
        self.relu=nn.ReLU(inplace=True)
        
        # Second 3x3 convolutional layer
        self.conv2=nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(out_channels)
        # self.relu=nn.ReLU(inplace=True)

        self.downsample=downsample

    def forward(self,x):
        identity=x
        if self.downsample is not None:
            identity = self.downsample(x)
        # Forward pass through the first convolutional layer
        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)

        # Forward pass through the second convolutional layer
        out=self.conv2(out)
        out=self.bn2(out)

        # Skip connection
        out+=identity
        out=self.relu(out)

        # return output
        return out

class ResNet34(nn.Module):
    def __init__(self, block, layers, num_classes=len(classes)):
        super(ResNet34,self).__init__()
        self.in_channels=64

        # Initial Convolutional Layer
        self.conv1=nn.Conv2d(3,64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1=nn.BatchNorm2d(64)
        self.relu=nn.ReLU(inplace=True)
        self.maxpool=nn.MaxPool2d(kernel_size=3, stride=2,padding=1)
        
        # Residual Layers
        self.layer1=self._make_layer(block, 64, layers[0], stride=1)
        self.layer2=self._make_layer(block, 128, layers[1], stride=2)
        self.layer3=self._make_layer(block, 256, layers[2], stride=2)
        self.layer4=self._make_layer(block, 512, layers[3], stride=2)

        # Final Output Layer
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.fc=nn.Linear(512, num_classes)

        # --- INITIALIZATION ---
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
        # This is the "Zero-init" trick for the last BN in each residual branch
        for m in self.modules():
            if isinstance(m, ResidualBlock):
                nn.init.constant_(m.bn2.weight, 0)


    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample=None
        if stride!=1 or self.in_channels!=out_channels:
            downsample=nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers=[]
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels=out_channels

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self,x):

        # Initial Convolutional Layer
        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)
        out=self.maxpool(out) 

        # Residual Layers
        out=self.layer1(out)
        out=self.layer2(out)
        out=self.layer3(out)
        out=self.layer4(out)

        # Final Output Layer
        out=self.avgpool(out)
        # out=out.view(out.size(0), -1)
        out = torch.flatten(out, 1) # Cleaner than .view()
        out=self.fc(out)
        return out
def resnet34():
    return ResNet34(ResidualBlock, [3, 4, 6, 3])

# Make the model

In [ ]:
# Check for Apple Silicon GPU (MPS)
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Using MPS (Apple Silicon GPU) 🚀")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
#     print("Using CUDA (NVIDIA GPU)")
# else:
#     device = torch.device("cpu")
#     print("Using CPU")

device=torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f'Using device: {device}')
# Instantiate the ResNet-34 model
model=resnet34().to(device)
# print(model)

## Defining the loss and optimizer

In [ ]:
# Define the loss function - CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# Define the optimizer with weight decay for L2 regularization - Adam Optimiser
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Every 7 epochs, multiply the learning rate by 0.1
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

# Use a more aggressive learning rate and momentum for better convergence on CIFAR-10
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
# Use a Cosine scheduler for a smooth decay over the full 50 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

## Training Loop

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# Set the number of epochs for training
num_epochs = 10

# Initialize the writer
writer = SummaryWriter('runs/resnet34_experiment')

# Add your model graph (Optional, shows you the ResNet structure)
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Training loop
for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train() 
    running_loss = 0.0
    for i, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if (i+1) % 100 == 0:
            avg_loss = running_loss / 100
            writer.add_scalar('Loss/train', avg_loss, epoch * len(trainloader) + i)
            running_loss = 0.0 # Reset for next 100 steps

    # --- VALIDATION PHASE (once per epoch) ---
    model.eval() # Set to evaluation mode (deactivates Dropout/BatchNorm)
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad(): # Disable gradient calculation (saves memory/time)
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(testloader)
    accuracy = 100 * correct / total

    # Update the Learning Rate Scheduler here
    scheduler.step()
    
    # Log validation metrics to TensorBoard
    writer.add_scalar('Loss/validation', avg_val_loss, epoch)
    writer.add_scalar('Accuracy/validation', accuracy, epoch)
    
    print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {accuracy:.2f}%')

writer.close()  # Close the writer after training is complete            
print("Training Complete!")

# Save the Model

In [ ]:
# Only saving the weights of the model, not the entire architecture, since we can easily reconstruct it with the code.  
torch.save(model.state_dict(), 'resnet34_cifar10_v1.pth')

# Loading the model
(and set it to evaluation mode for inference)

In [ ]:
# Load the structure first and then the weights.
load_model=resnet34()
load_model.load_state_dict(torch.load('resnet34_cifar10_v1.pth'))

# Important steps for M3 inference.
load_model.to(device)
load_model.eval() # Set to evaluation mode

NameError: name 'resnet34' is not defined